In [1]:
!pip install catboost
!pip install optuna catboost scikit-learn optunahub -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, accuracy_score, recall_score,
                             f1_score, precision_score, roc_auc_score, confusion_matrix,
                             roc_curve, auc, precision_recall_curve, ConfusionMatrixDisplay,
                             average_precision_score)
from sklearn.utils import resample

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import optuna
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
from catboost import CatBoostClassifier
import warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.7 MB/s eta 0:00:00


In [2]:
def evaluate_model(y_true, y_pred, y_proba, model_name="Model"):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  #Recall
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision   = tp / (tp + fp) if (tp + fp) > 0 else 0

    acc   = accuracy_score(y_true, y_pred)
    f1    = f1_score(y_true, y_pred, zero_division=0)
    auc_roc = roc_auc_score(y_true, y_proba)
    auprc   = average_precision_score(y_true, y_proba)

    print(f"========== PERFORMANTA: {model_name} ==========")
    print(f"AUC-ROC:     {auc_roc:.4f}")
    print(f"AUPRC:       {auprc:.4f}")
    print(f"Accuracy:    {acc:.4f}")
    print(f"F1-Score:    {f1:.4f}")
    print(f"Sensitivity: {sensitivity:.4f} (Recall / True Positive Rate)")
    print(f"Specificity: {specificity:.4f} (True Negative Rate)")
    print(f"Precision:   {precision:.4f} (Positive Predictive Value)")
    print("-------------------------------------------------")
    print("Confusion Matrix:")
    print(f"[{tn}] TN   [{fp}] FP")
    print(f"[{fn}] FN   [{tp}] TP\n")

    return {'auc': auc_roc, 'auprc': auprc, 'acc': acc, 'f1': f1,
            'sens': sensitivity, 'spec': specificity, 'prec': precision}

In [3]:
data = pd.read_excel('/content/dataset.xlsx')
print("Shape: ", data.shape)

features_df    = pd.read_csv('/content/FINAL_35_features_selected.csv')
FINAL_FEATURES = features_df['feature'].tolist()
N_FEATURES     = len(FINAL_FEATURES)

X = data[FINAL_FEATURES]
y = data['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
print("Distributia datelor:")
print(f"X_train shape: {X_train.shape} ({X_train.shape[0]} paciente, {X_train.shape[1]} simptome)")
print(f"X_test shape:  {X_test.shape} ({X_test.shape[0]} paciente, {X_test.shape[1]} simptome)")
print(f"Prevalența bolii (Train): {y_train.mean()*100:.1f}%")
print(f"Prevalența bolii (Test):  {y_test.mean()*100:.1f}%")

Shape:  (886, 61)
Distributia datelor:
X_train shape: (708, 35) (708 paciente, 35 simptome)
X_test shape:  (178, 35) (178 paciente, 35 simptome)
Prevalența bolii (Train): 53.5%
Prevalența bolii (Test):  53.4%


In [4]:
def evaluate_model_full(y_true, y_pred, y_proba, model_name="Model"):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    metrics = {
        "accuracy"    : accuracy_score(y_true, y_pred),
        "roc_auc"     : roc_auc_score(y_true, y_proba),
        "auprc"       : average_precision_score(y_true, y_proba),
        "f1"          : f1_score(y_true, y_pred, zero_division=0),
        "precision"   : precision_score(y_true, y_pred, zero_division=0),
        "recall"      : recall_score(y_true, y_pred, zero_division=0),   # sensitivity
        "specificity" : tn / (tn + fp) if (tn + fp) > 0 else 0.0,
    }

    print(f"\n{'='*55}")
    print(f" {model_name}")
    print(f"{'='*55}")
    print(f"  Accuracy        : {metrics['accuracy']:.4f}")
    print(f"  AUC-ROC         : {metrics['roc_auc']:.4f}")
    print(f"  AUPRC           : {metrics['auprc']:.4f}")
    print(f"  F1-Score        : {metrics['f1']:.4f}")
    print(f"  Precision       : {metrics['precision']:.4f}")
    print(f"  Recall/Sensitiv : {metrics['recall']:.4f}")
    print(f"  Specificity     : {metrics['specificity']:.4f}")
    print(f"{'='*55}")
    print(f"\n  Confusion Matrix:\n  TN={tn}  FP={fp}\n  FN={fn}  TP={tp}")
    print(f"\n{classification_report(y_true, y_pred, zero_division=0)}")

    return metrics

HYPERPARAMS TUNNING

In [ ]:
# RANDOM FOREST — NESTED CV + OPTUNA
import optuna
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, classification_report
)
import warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

OUTER_FOLDS_RF  = 10
INNER_FOLDS_RF  = 5
N_TRIALS_RF     = 60
N_TRIALS_FINAL  = 100
RANDOM_STATE    = 42
SCORING         = "roc_auc"

FIXED_PARAMS = {
    "class_weight" : "balanced",
    "bootstrap"    : True,
    "random_state" : RANDOM_STATE,
    "n_jobs"       : -1,
}

outer_cv_rf = StratifiedKFold(n_splits=OUTER_FOLDS_RF, shuffle=True, random_state=RANDOM_STATE)
inner_cv_rf = StratifiedKFold(n_splits=INNER_FOLDS_RF, shuffle=True, random_state=RANDOM_STATE)

def objective_rf(trial, X_tr, y_tr, cv):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 100, 1500, step=100),
        "criterion"        : trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "max_depth"        : trial.suggest_int("max_depth", 3, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
        "min_samples_leaf" : trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features"     : trial.suggest_categorical(
                                 "max_features", ["sqrt", "log2", 0.3, 0.5, 0.7]
                             ),
        "max_samples"      : trial.suggest_float("max_samples", 0.5, 1.0),
        **FIXED_PARAMS,
    }
    model  = RandomForestClassifier(**params)
    scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring=SCORING, n_jobs=1)
    return scores.mean()

outer_scores_rf      = []
best_params_rf_folds = []

X_arr = X.values if hasattr(X, 'values') else X
y_arr = y.values if hasattr(y, 'values') else y

print("=" * 60)
print(f"  RANDOM FOREST — NESTED CV")
print(f"  ({OUTER_FOLDS_RF}-outer × {INNER_FOLDS_RF}-inner × {N_TRIALS_RF} trials)")
print("=" * 60)

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv_rf.split(X_arr, y_arr), 1):
    X_outer_train, X_outer_test = X_arr[train_idx], X_arr[test_idx]
    y_outer_train, y_outer_test = y_arr[train_idx], y_arr[test_idx]

    study_rf = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study_rf.optimize(
        lambda trial: objective_rf(trial, X_outer_train, y_outer_train, inner_cv_rf),
        n_trials=N_TRIALS_RF,
        show_progress_bar=False
    )

    best_p = study_rf.best_params
    best_params_rf_folds.append(best_p)

    final_rf = RandomForestClassifier(**best_p, **FIXED_PARAMS)
    final_rf.fit(X_outer_train, y_outer_train)

    y_proba_fold = final_rf.predict_proba(X_outer_test)[:, 1]
    auc_fold     = roc_auc_score(y_outer_test, y_proba_fold)
    outer_scores_rf.append(auc_fold)

    print(f"  Fold {fold_idx:2d}/10 | Inner AUC: {study_rf.best_value:.4f} "
          f"| Test AUC: {auc_fold:.4f} "
          f"| n_est={best_p.get('n_estimators')} "
          f"depth={best_p.get('max_depth')} "
          f"feat={best_p.get('max_features')}")

print("=" * 60)
print(f"  NESTED CV AUC-ROC: {np.mean(outer_scores_rf):.4f} ± {np.std(outer_scores_rf):.4f}")
print(f"  Min: {np.min(outer_scores_rf):.4f}  |  Max: {np.max(outer_scores_rf):.4f}")
print("=" * 60)


print(f"\n Studiu Optuna final pe tot X_train ({N_TRIALS_FINAL} trials)...")

final_inner_cv = StratifiedKFold(n_splits=INNER_FOLDS_RF, shuffle=True, random_state=RANDOM_STATE)

study_final = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_final.optimize(
    lambda trial: objective_rf(trial, X_train, y_train, final_inner_cv),
    n_trials=N_TRIALS_FINAL,
    show_progress_bar=True
)

FINAL_PARAMS_RF = study_final.best_params
print(f"\n Parametrii finali: {FINAL_PARAMS_RF}")
print(f"   Best CV AUC (inner): {study_final.best_value:.4f}")

best_rf_tuned = RandomForestClassifier(**FINAL_PARAMS_RF, **FIXED_PARAMS)
best_rf_tuned.fit(X_train, y_train)

y_pred_rf_tuned  = best_rf_tuned.predict(X_test)
y_proba_rf_tuned = best_rf_tuned.predict_proba(X_test)[:, 1]

rf_tuned_metrics = evaluate_model_full(
    y_test, y_pred_rf_tuned, y_proba_rf_tuned,
    model_name="Random Forest (Optuna Tuned — Final)"
)

print(f"\n COMPARATIE RF:")
print(f"   {'Metric':<20} {'Default':>10} {'Tuned':>10} {'Δ':>10}")
print(f"   {'-'*50}")
for metric in ["accuracy", "roc_auc", "auprc", "f1", "precision", "recall", "specificity"]:
    default_val = rf_test_metrics.get(metric, 0)
    tuned_val   = rf_tuned_metrics.get(metric, 0)
    delta       = tuned_val - default_val
    sign        = "+" if delta >= 0 else ""
    print(f"   {metric:<20} {default_val:>10.4f} {tuned_val:>10.4f} {sign}{delta:>9.4f}")

  RANDOM FOREST — NESTED CV
  (10-outer × 5-inner × 60 trials)
  Fold  1/10 | Inner AUC: 0.9756 | Test AUC: 0.9873 | n_est=700 depth=22 feat=sqrt
  Fold  2/10 | Inner AUC: 0.9756 | Test AUC: 0.9726 | n_est=1400 depth=38 feat=sqrt
  Fold  3/10 | Inner AUC: 0.9735 | Test AUC: 0.9837 | n_est=600 depth=23 feat=sqrt
  Fold  4/10 | Inner AUC: 0.9735 | Test AUC: 0.9698 | n_est=700 depth=31 feat=sqrt
  Fold  5/10 | Inner AUC: 0.9758 | Test AUC: 0.9612 | n_est=200 depth=36 feat=sqrt
  Fold  6/10 | Inner AUC: 0.9721 | Test AUC: 0.9904 | n_est=800 depth=22 feat=sqrt
  Fold  7/10 | Inner AUC: 0.9752 | Test AUC: 0.9694 | n_est=900 depth=19 feat=sqrt
  Fold  8/10 | Inner AUC: 0.9738 | Test AUC: 0.9927 | n_est=1100 depth=39 feat=sqrt
  Fold  9/10 | Inner AUC: 0.9780 | Test AUC: 0.9606 | n_est=800 depth=23 feat=sqrt
  Fold 10/10 | Inner AUC: 0.9756 | Test AUC: 0.9683 | n_est=1100 depth=33 feat=sqrt
  NESTED CV AUC-ROC: 0.9756 ± 0.0113
  Min: 0.9606  |  Max: 0.9927

🔍 Studiu Optuna final pe tot X_train

  0%|          | 0/100 [00:00<?, ?it/s]


✅ Parametrii finali: {'n_estimators': 1000, 'criterion': 'gini', 'max_depth': 27, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.3, 'max_samples': 0.8322097211712385}
   Best CV AUC (inner): 0.9761

  📊 Random Forest (Optuna Tuned — Final)
  Accuracy        : 0.9101
  AUC-ROC         : 0.9721
  AUPRC           : 0.9786
  F1-Score        : 0.9158
  Precision       : 0.9158
  Recall/Sensitiv : 0.9158
  Specificity     : 0.9036

  Confusion Matrix:
  TN=75  FP=8
  FN=8  TP=87

              precision    recall  f1-score   support

           0       0.90      0.90      0.90        83
           1       0.92      0.92      0.92        95

    accuracy                           0.91       178
   macro avg       0.91      0.91      0.91       178
weighted avg       0.91      0.91      0.91       178


📈 COMPARATIE RF:
   Metric                  Default      Tuned          Δ
   --------------------------------------------------


NameError: name 'rf_test_metrics' is not defined

In [ ]:

# RANDOM FOREST — NESTED CV + OPTUNA
import optuna
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, classification_report
)
import warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

# CONFIG
OUTER_FOLDS_RF  = 10
INNER_FOLDS_RF  = 5
N_TRIALS_RF     = 80
N_TRIALS_FINAL  = 120
RANDOM_STATE    = 42
SCORING         = "roc_auc"

FIXED_PARAMS = {
    "class_weight" : "balanced",
    "bootstrap"    : True,
    "random_state" : RANDOM_STATE,
    "n_jobs"       : -1,
}

outer_cv_rf = StratifiedKFold(n_splits=OUTER_FOLDS_RF, shuffle=True, random_state=RANDOM_STATE)
inner_cv_rf = StratifiedKFold(n_splits=INNER_FOLDS_RF, shuffle=True, random_state=RANDOM_STATE)

# EVALUARE COMPLETA
def evaluate_model_full(y_true, y_pred, y_proba, model_name="Model"):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    metrics = {
        "accuracy"    : accuracy_score(y_true, y_pred),
        "roc_auc"     : roc_auc_score(y_true, y_proba),
        "auprc"       : average_precision_score(y_true, y_proba),
        "f1"          : f1_score(y_true, y_pred, zero_division=0),
        "precision"   : precision_score(y_true, y_pred, zero_division=0),
        "recall"      : recall_score(y_true, y_pred, zero_division=0),
        "specificity" : tn / (tn + fp) if (tn + fp) > 0 else 0.0,
    }
    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    for k, v in metrics.items():
        print(f"  {k:<18}: {v:.4f}")
    print(f"{'='*55}")
    print(f"\n  Confusion Matrix:\n  TN={tn}  FP={fp}\n  FN={fn}  TP={tp}")
    print(f"\n{classification_report(y_true, y_pred, zero_division=0)}")
    return metrics

# OBIECTIV OPTUNA
def objective_rf(trial, X_tr, y_tr, cv):
    use_max_depth = trial.suggest_categorical("use_max_depth", [True, False])
    max_depth = trial.suggest_int("max_depth", 10, 50) if use_max_depth else None

    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 100, 800, step=50),
        "criterion"        : trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "max_depth"        : max_depth,
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf" : trial.suggest_int("min_samples_leaf", 1, 8),
        "max_features"     : trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.7, None]),
        "max_samples"      : trial.suggest_float("max_samples", 0.7, 1.0),
        **FIXED_PARAMS,
    }
    model  = RandomForestClassifier(**params)
    scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring=SCORING, n_jobs=1)
    return scores.mean()

# NESTED CV
outer_scores_rf      = []
best_params_rf_folds = []

X_arr = X.values if hasattr(X, 'values') else X
y_arr = y.values if hasattr(y, 'values') else y

print(f"  RANDOM FOREST — NESTED CV (FIXED)")
print(f"  ({OUTER_FOLDS_RF}-outer × {INNER_FOLDS_RF}-inner × {N_TRIALS_RF} trials)")

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv_rf.split(X_arr, y_arr), 1):
    X_outer_train, X_outer_test = X_arr[train_idx], X_arr[test_idx]
    y_outer_train, y_outer_test = y_arr[train_idx], y_arr[test_idx]

    study_rf = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE + fold_idx)  # different seed per fold
    )
    study_rf.optimize(
        lambda trial: objective_rf(trial, X_outer_train, y_outer_train, inner_cv_rf),
        n_trials=N_TRIALS_RF,
        show_progress_bar=False
    )

    best_p = study_rf.best_params.copy()

    if not best_p.pop("use_max_depth"):
        best_p["max_depth"] = None

    best_params_rf_folds.append(best_p)

    final_rf = RandomForestClassifier(**best_p, **FIXED_PARAMS)
    final_rf.fit(X_outer_train, y_outer_train)

    y_proba_fold = final_rf.predict_proba(X_outer_test)[:, 1]
    auc_fold     = roc_auc_score(y_outer_test, y_proba_fold)
    outer_scores_rf.append(auc_fold)

    depth_str = best_p.get('max_depth') or 'None'
    print(f"  Fold {fold_idx:2d}/10 | Inner AUC: {study_rf.best_value:.4f} "
          f"| Test AUC: {auc_fold:.4f} "
          f"| n_est={best_p.get('n_estimators')} "
          f"depth={depth_str} "
          f"feat={best_p.get('max_features')}")

print("=" * 60)
print(f"  NESTED CV AUC-ROC: {np.mean(outer_scores_rf):.4f} ± {np.std(outer_scores_rf):.4f}")
print(f"  Min: {np.min(outer_scores_rf):.4f}  |  Max: {np.max(outer_scores_rf):.4f}")
print("=" * 60)

# STUDIU FINAL
print(f"\n Studiu Optuna final pe tot X_train ({N_TRIALS_FINAL} trials)...")

final_inner_cv = StratifiedKFold(n_splits=INNER_FOLDS_RF, shuffle=True, random_state=RANDOM_STATE)

study_final = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_final.optimize(
    lambda trial: objective_rf(trial, X_train, y_train, final_inner_cv),
    n_trials=N_TRIALS_FINAL,
    show_progress_bar=True
)

FINAL_PARAMS_RF = study_final.best_params.copy()
if not FINAL_PARAMS_RF.pop("use_max_depth"):
    FINAL_PARAMS_RF["max_depth"] = None

print(f"\n Parametrii finali: {FINAL_PARAMS_RF}")
print(f"   Best CV AUC (inner): {study_final.best_value:.4f}")

# MODEL FINAL
best_rf_tuned = RandomForestClassifier(**FINAL_PARAMS_RF, **FIXED_PARAMS)
best_rf_tuned.fit(X_train, y_train)

y_pred_rf_tuned  = best_rf_tuned.predict(X_test)
y_proba_rf_tuned = best_rf_tuned.predict_proba(X_test)[:, 1]

rf_tuned_metrics = evaluate_model_full(
    y_test, y_pred_rf_tuned, y_proba_rf_tuned,
    model_name="Random Forest (Optuna Tuned — Fixed)"
)

# COMPARATIE
print(f"\nCOMPARATIE RF:")
print(f"   {'Metric':<20} {'Default':>10} {'Tuned':>10} {'Δ':>10}")
rf_test_metrics = {
    "roc_auc"     : 0.976,
    "auprc"       : 0.981,
    "f1"          : 0.936,
    "precision"   : 0.926,
    "recall"      : 0.940,
    "specificity" : 0.946,
    "accuracy"    : 0.933,
}

for metric in ["accuracy", "roc_auc", "auprc", "f1", "precision", "recall", "specificity"]:
    default_val = rf_test_metrics.get(metric, 0)
    tuned_val   = rf_tuned_metrics.get(metric, 0)
    delta       = tuned_val - default_val
    sign        = "+" if delta >= 0 else ""
    print(f"   {metric:<20} {default_val:>10.4f} {tuned_val:>10.4f} {sign}{delta:>9.4f}")


  RANDOM FOREST — NESTED CV (FIXED)
  (10-outer × 5-inner × 80 trials)
  Fold  1/10 | Inner AUC: 0.9760 | Test AUC: 0.9876 | n_est=350 depth=None feat=log2
  Fold  2/10 | Inner AUC: 0.9746 | Test AUC: 0.9736 | n_est=250 depth=None feat=sqrt
  Fold  3/10 | Inner AUC: 0.9742 | Test AUC: 0.9888 | n_est=400 depth=None feat=log2
  Fold  4/10 | Inner AUC: 0.9748 | Test AUC: 0.9705 | n_est=200 depth=27 feat=log2
  Fold  5/10 | Inner AUC: 0.9770 | Test AUC: 0.9607 | n_est=650 depth=33 feat=sqrt
  Fold  6/10 | Inner AUC: 0.9726 | Test AUC: 0.9878 | n_est=800 depth=None feat=sqrt
  Fold  7/10 | Inner AUC: 0.9755 | Test AUC: 0.9720 | n_est=650 depth=None feat=log2
  Fold  8/10 | Inner AUC: 0.9742 | Test AUC: 0.9891 | n_est=150 depth=37 feat=sqrt
  Fold  9/10 | Inner AUC: 0.9784 | Test AUC: 0.9559 | n_est=800 depth=15 feat=log2
  Fold 10/10 | Inner AUC: 0.9768 | Test AUC: 0.9699 | n_est=500 depth=18 feat=log2
  NESTED CV AUC-ROC: 0.9756 ± 0.0116
  Min: 0.9559  |  Max: 0.9891

 Studiu Optuna final 

  0%|          | 0/120 [00:00<?, ?it/s]


 Parametrii finali: {'n_estimators': 350, 'criterion': 'entropy', 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_samples': 0.942379993301536, 'max_depth': None}
   Best CV AUC (inner): 0.9777

  Random Forest (Optuna Tuned — Fixed)
  accuracy          : 0.9101
  roc_auc           : 0.9736
  auprc             : 0.9793
  f1                : 0.9149
  precision         : 0.9247
  recall            : 0.9053
  specificity       : 0.9157

  Confusion Matrix:
  TN=76  FP=7
  FN=9  TP=86

              precision    recall  f1-score   support

           0       0.89      0.92      0.90        83
           1       0.92      0.91      0.91        95

    accuracy                           0.91       178
   macro avg       0.91      0.91      0.91       178
weighted avg       0.91      0.91      0.91       178


COMPARATIE RF:
   Metric                  Default      Tuned          Δ
   accuracy                 0.9330     0.9101   -0.0229
   roc_auc                  0

**DEFAULT+YOUNDEN**

In [8]:
rf_default = RandomForestClassifier(
    n_estimators = 100,
    class_weight = "balanced",
    random_state = 42,
    n_jobs       = -1
)
cv_outer_rf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_rf = cross_val_score(
    rf_default, X_train, y_train,
    cv      = cv_outer_rf,
    scoring = "roc_auc",
    n_jobs  = -1
)
print(f"10-Fold CV AUC-ROC: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}\n")

rf_default.fit(X_train, y_train)
y_proba_train_oof = cross_val_predict(
    rf_default, X_train, y_train,
    cv      = cv_outer_rf,
    method  = 'predict_proba',
    n_jobs  = -1
)[:, 1]
fpr_train, tpr_train, thresholds_train = roc_curve(y_train, y_proba_train_oof)
best_thresh_rf = thresholds_train[np.argmax(tpr_train - fpr_train)]
y_proba_rf_def = rf_default.predict_proba(X_test)[:, 1]
y_pred_rf_05 = (y_proba_rf_def >= 0.5).astype(int)
metrics_rf_05 = evaluate_model_full(
    y_test, y_pred_rf_05, y_proba_rf_def,
    model_name="Random Forest Default (thresh=0.50)"
)
print(f"\n  Threshold default : 0.5000")
print(f"  Threshold Youden  : {best_thresh_rf:.4f} (calculat pe datele de Train)")

y_pred_rf_youden = (y_proba_rf_def >= best_thresh_rf).astype(int)
metrics_rf_youden = evaluate_model_full(
    y_test, y_pred_rf_youden, y_proba_rf_def,
    model_name=f"Random Forest Default + Youden (thresh={best_thresh_rf:.4f})"
)

print(f"\n COMPARISON Random Forest Default:")
print(f"   {'Metric':<20} {'thresh=0.50':>12} {'Youden':>12} {'Δ':>10}")
for metric in ["accuracy", "roc_auc", "auprc", "f1", "precision", "recall", "specificity"]:
    v05     = metrics_rf_05.get(metric, 0)
    vyouden = metrics_rf_youden.get(metric, 0)
    delta   = vyouden - v05
    sign    = "+" if delta >= 0 else ""
    print(f"   {metric:<20} {v05:>12.4f} {vyouden:>12.4f} {sign}{delta:>9.4f}")

model_bundle = {
    'model': rf_default,
    'threshold': best_thresh_rf,
    'features': X_train.columns.tolist() if hasattr(X_train, 'columns') else None
}
nume_fisier = "rf_model_youden.joblib"
joblib.dump(model_bundle, nume_fisier)

if colab_available:
    files.download(nume_fisier)

10-Fold CV AUC-ROC: 0.9709 ± 0.0151


 Random Forest Default (thresh=0.50)
  Accuracy        : 0.9270
  AUC-ROC         : 0.9762
  AUPRC           : 0.9807
  F1-Score        : 0.9312
  Precision       : 0.9362
  Recall/Sensitiv : 0.9263
  Specificity     : 0.9277

  Confusion Matrix:
  TN=77  FP=6
  FN=7  TP=88

              precision    recall  f1-score   support

           0       0.92      0.93      0.92        83
           1       0.94      0.93      0.93        95

    accuracy                           0.93       178
   macro avg       0.93      0.93      0.93       178
weighted avg       0.93      0.93      0.93       178


  Threshold default : 0.5000
  Threshold Youden  : 0.5377 (calculat pe datele de Train)

 Random Forest Default + Youden (thresh=0.5377)
  Accuracy        : 0.9326
  AUC-ROC         : 0.9762
  AUPRC           : 0.9807
  F1-Score        : 0.9355
  Precision       : 0.9560
  Recall/Sensitiv : 0.9158
  Specificity     : 0.9518

  Confusion Matrix:
  TN=79  FP

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**OPTUNA+NestedCV**

In [ ]:
import optuna
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, make_scorer
)
import warnings
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

RANDOM_STATE   = 42
OUTER_FOLDS    = 10
INNER_FOLDS    = 5
N_TRIALS       = 100
N_TRIALS_FINAL = 200

outer_cv = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)

FIXED_PARAMS_RF = {
    "class_weight" : "balanced",
    "bootstrap"    : True,
    "random_state" : RANDOM_STATE,
    "n_jobs"       : 1
}

def objective_rf(trial, X_tr, y_tr):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 100, 800, step=50),
        "criterion"        : trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "max_depth"        : trial.suggest_int("max_depth", 3, 25),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf" : trial.suggest_int("min_samples_leaf", 1, 15),
        "max_features"     : trial.suggest_categorical(
                                 "max_features", ["sqrt", "log2", 0.3, 0.5, 0.7]
                             ),
        "max_samples"      : trial.suggest_float("max_samples", 0.5, 1.0),
        **FIXED_PARAMS_RF,
    }

    model  = RandomForestClassifier(**params)
    scores = cross_val_score(
        model, X_tr, y_tr,
        cv          = inner_cv,
        scoring     = "roc_auc",
        n_jobs      = 1,
        error_score = 0.0
    )
    return float(scores.mean())

outer_scores_rf      = []
best_params_rf_folds = []

X_arr = X_train.values if hasattr(X_train, 'values') else X_train
y_arr = y_train.values if hasattr(y_train, 'values') else y_train

print(f"  RANDOM FOREST — NESTED CV")
print(f"  {OUTER_FOLDS}-outer × {INNER_FOLDS}-inner × {N_TRIALS} trials")
print(f"  Scoring: AUC-ROC | Youden la final")

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_arr, y_arr), 1):
    X_outer_train, X_outer_test = X_arr[train_idx], X_arr[test_idx]
    y_outer_train, y_outer_test = y_arr[train_idx], y_arr[test_idx]

    study_rf = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed             = RANDOM_STATE + fold_idx,
            n_startup_trials = 10,
            multivariate     = True,
            warn_independent_sampling = False
        )
    )
    study_rf.optimize(
        lambda trial: objective_rf(trial, X_outer_train, y_outer_train),
        n_trials          = N_TRIALS,
        show_progress_bar = False
    )

    best_p = study_rf.best_params
    best_params_rf_folds.append(best_p)

    final_rf = RandomForestClassifier(**best_p, **FIXED_PARAMS_RF)
    final_rf.fit(X_outer_train, y_outer_train)

    y_proba_fold = final_rf.predict_proba(X_outer_test)[:, 1]
    auc_fold     = roc_auc_score(y_outer_test, y_proba_fold)
    outer_scores_rf.append(auc_fold)

    print(f"  Fold {fold_idx:2d}/{OUTER_FOLDS} | Inner AUC: {study_rf.best_value:.4f} "
          f"| Test AUC: {auc_fold:.4f} "
          f"| n_est={best_p.get('n_estimators')} "
          f"| depth={best_p.get('max_depth')} "
          f"| feat={best_p.get('max_features')}")

print(f"  NESTED CV AUC-ROC : {np.mean(outer_scores_rf):.4f} ± {np.std(outer_scores_rf):.4f}")
print(f"  Min: {np.min(outer_scores_rf):.4f}  |  Max: {np.max(outer_scores_rf):.4f}")

study_final_rf = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed             = RANDOM_STATE,
        n_startup_trials = 20,
        multivariate     = True,
        warn_independent_sampling = False
    )
)
study_final_rf.optimize(
    lambda trial: objective_rf(trial, X_train, y_train),
    n_trials          = N_TRIALS_FINAL,
    show_progress_bar = True
)

FINAL_PARAMS_RF = study_final_rf.best_params
print(f"\n Parametrii finali: {FINAL_PARAMS_RF}")
print(f"   Best CV AUC (inner): {study_final_rf.best_value:.4f}")

best_rf_tuned = RandomForestClassifier(**FINAL_PARAMS_RF, **FIXED_PARAMS_RF)
best_rf_tuned.fit(X_train, y_train)

y_proba_rf_tuned = best_rf_tuned.predict_proba(X_test)[:, 1]
y_pred_rf_tuned  = best_rf_tuned.predict(X_test)

rf_tuned_metrics = evaluate_model_full(
    y_test, y_pred_rf_tuned, y_proba_rf_tuned,
    model_name="RF Tuned (majority vote)"
)

# ── YOUDEN THRESHOLD
fpr, tpr, thresholds = roc_curve(y_test, y_proba_rf_tuned)
best_thresh_rf = thresholds[np.argmax(tpr - fpr)]

print(f"\n  Threshold Youden : {best_thresh_rf:.4f}")

y_pred_rf_youden = (y_proba_rf_tuned >= best_thresh_rf).astype(int)

rf_youden_metrics = evaluate_model_full(
    y_test, y_pred_rf_youden, y_proba_rf_tuned,
    model_name=f"RF Tuned + Youden (thresh={best_thresh_rf:.4f})"
)

rf_baseline_metrics = {
    "accuracy"    : 0.933,
    "roc_auc"     : 0.976,
    "auprc"       : 0.981,
    "f1"          : 0.936,
    "precision"   : 0.946,
    "recall"      : 0.926,
    "specificity" : 0.940,
}

print(f"\n  COMPARATIE FINALA RANDOM FOREST:")
print(f"   {'Metric':<20} {'Baseline':>10} {'Tuned':>10} {'Youden':>10} {'Δ base→Youden':>16}")
print(f"   {'-'*62}")
for metric in ["accuracy", "roc_auc", "auprc", "f1", "precision", "recall", "specificity"]:
    b = rf_baseline_metrics.get(metric, 0)
    t = rf_tuned_metrics.get(metric, 0)
    y = rf_youden_metrics.get(metric, 0)
    delta = y - b
    sign  = "+" if delta >= 0 else ""
    print(f"   {metric:<20} {b:>10.4f} {t:>10.4f} {y:>10.4f} {sign}{delta:>15.4f}")

  RANDOM FOREST — NESTED CV
  10-outer × 5-inner × 100 trials
  Scoring: AUC-ROC | Youden la final
  Fold  1/10 | Inner AUC: 0.9786 | Test AUC: 0.9569 | n_est=550 | depth=20 | feat=log2
  Fold  2/10 | Inner AUC: 0.9751 | Test AUC: 0.9729 | n_est=200 | depth=18 | feat=log2
  Fold  3/10 | Inner AUC: 0.9715 | Test AUC: 0.9888 | n_est=150 | depth=23 | feat=sqrt
  Fold  4/10 | Inner AUC: 0.9726 | Test AUC: 0.9916 | n_est=500 | depth=20 | feat=sqrt
  Fold  5/10 | Inner AUC: 0.9759 | Test AUC: 0.9641 | n_est=800 | depth=14 | feat=sqrt
  Fold  6/10 | Inner AUC: 0.9759 | Test AUC: 0.9601 | n_est=500 | depth=19 | feat=0.3
  Fold  7/10 | Inner AUC: 0.9719 | Test AUC: 0.9769 | n_est=100 | depth=25 | feat=log2
  Fold  8/10 | Inner AUC: 0.9762 | Test AUC: 0.9729 | n_est=250 | depth=17 | feat=log2
  Fold  9/10 | Inner AUC: 0.9756 | Test AUC: 0.9523 | n_est=300 | depth=17 | feat=log2
  Fold 10/10 | Inner AUC: 0.9687 | Test AUC: 0.9967 | n_est=750 | depth=19 | feat=0.7
  NESTED CV AUC-ROC : 0.9733 ± 0.

  0%|          | 0/200 [00:00<?, ?it/s]


 Parametrii finali: {'n_estimators': 500, 'criterion': 'entropy', 'max_depth': 20, 'min_samples_split': 3, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_samples': 0.8027351761371895}
   Best CV AUC (inner): 0.9776
  RF Tuned (majority vote)
  Accuracy         : 0.9045
  AUC-ROC          : 0.9729
  AUPRC            : 0.9788
  F1-Score         : 0.9101
  Precision        : 0.9149
  Recall/Sensitiv  : 0.9053
  Specificity      : 0.9036

  Confusion Matrix:
  TN=75  FP=8
  FN=9  TP=86

              precision    recall  f1-score   support

           0       0.89      0.90      0.90        83
           1       0.91      0.91      0.91        95

    accuracy                           0.90       178
   macro avg       0.90      0.90      0.90       178
weighted avg       0.90      0.90      0.90       178


  Threshold Youden : 0.5913
  RF Tuned + Youden (thresh=0.5913)
  Accuracy         : 0.9213
  AUC-ROC          : 0.9729
  AUPRC            : 0.9788
  F1-Score         : 0.9239
  

BOOTRAP

In [9]:
rf_model = RandomForestClassifier(
    n_estimators=100, class_weight="balanced",
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)
YOUDEN_THRESH = best_thresh_rf #0.5377
print(f"Using fixed Youden threshold: {YOUDEN_THRESH:}\n")

y_proba = rf_model.predict_proba(X_test)[:, 1]
N_BOOT = 1000
np.random.seed(42)

boot_metrics = {m: [] for m in
    ["accuracy", "roc_auc", "auprc", "f1",
     "precision", "recall", "specificity"]}
y_test_arr = np.asarray(y_test)

for i in range(N_BOOT):
    idx = resample(np.arange(len(y_test_arr)),
                   stratify=y_test_arr, random_state=i)
    y_true_b  = y_test_arr[idx]
    y_proba_b = y_proba[idx]
    y_pred_b  = (y_proba_b >= YOUDEN_THRESH).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true_b, y_pred_b).ravel()

    boot_metrics["accuracy"].append(accuracy_score(y_true_b, y_pred_b))
    boot_metrics["roc_auc"].append(roc_auc_score(y_true_b, y_proba_b))
    boot_metrics["auprc"].append(average_precision_score(y_true_b, y_proba_b))
    boot_metrics["f1"].append(f1_score(y_true_b, y_pred_b))
    boot_metrics["precision"].append(precision_score(y_true_b, y_pred_b))
    boot_metrics["recall"].append(recall_score(y_true_b, y_pred_b))
    boot_metrics["specificity"].append(tn/(tn+fp) if (tn+fp) > 0 else 0)

print(f"  BOOTSTRAP 95% CI RF Default + Youden (n={N_BOOT})")
print(f"  {'Metric':<15} {'Mean':>8} {'95% CI':>22}")
for m in boot_metrics:
    vals = np.array(boot_metrics[m])
    lo = np.percentile(vals, 2.5)
    hi = np.percentile(vals, 97.5)
    print(f"  {m:<15} {vals.mean():>8.4f}   [{lo:.4f} — {hi:.4f}]")

Using fixed Youden threshold: 0.5376599065958672

  BOOTSTRAP 95% CI RF Default + Youden (n=1000)
  Metric              Mean                 95% CI
  accuracy          0.9329   [0.8933 — 0.9663]
  roc_auc           0.9763   [0.9560 — 0.9909]
  auprc             0.9808   [0.9652 — 0.9923]
  f1                0.9358   [0.8973 — 0.9684]
  precision         0.9560   [0.9158 — 0.9889]
  recall            0.9170   [0.8526 — 0.9684]
  specificity       0.9512   [0.9036 — 0.9880]
